# Stage 1 Checkpoint Selection: Held-Out Evaluation

Evaluation-only companion to notebook 5. This compares saved Stage 1A and Stage 1B autoencoder checkpoints on fixed held-out test splits and writes selection tables, per-example metrics, split fingerprints, plots, and the final downstream AE checkpoint manifest.

No model training occurs in this notebook. Its main output is the selected-checkpoint manifest consumed by notebooks 6 and 6a.


In [ ]:
from pathlib import Path
import os
import platform
import shutil
import subprocess
import sys

print("Python:", sys.version)
print("Platform:", platform.platform())

# Match notebook 6's Colab setup: code repo in /content/neurovlm_gnn, Drive for data/run outputs.
REPO_URL = os.environ.get("NEUROVLM_REPO_URL", "https://github.com/neurovlm/neurovlm.git")
REPO_BRANCH = os.environ.get("NEUROVLM_REPO_BRANCH", "neurovlm_gnn")
REPO_DIR = Path(os.environ.get("NEUROVLM_REPO_DIR", "/content/neurovlm_gnn"))
DRIVE_ROOT = Path(os.environ.get("NEUROVLM_DRIVE_ROOT", "/content/drive/MyDrive/neurovlm"))
INSTALL_DEPENDENCIES = os.environ.get("NEUROVLM_INSTALL_DEPS", "1") == "1"
LOCK_DOWNSTREAM_STAGE1_SELECTION = os.environ.get("NEUROVLM_LOCK_DOWNSTREAM_STAGE1_SELECTION", "1") == "1"
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
else:
    # Local fallback for running inside this workspace.
    local_repo = Path.cwd()
    if (local_repo / "experiments/3dcnn").exists():
        REPO_DIR = local_repo
        DRIVE_ROOT = Path(os.environ.get("NEUROVLM_DRIVE_ROOT", str(local_repo)))


def run_cmd(cmd, cwd=None, *, check=True):
    print("$", " ".join(map(str, cmd)))
    result = subprocess.run(cmd, cwd=cwd, text=True, capture_output=True)
    if result.stdout.strip():
        print(result.stdout.strip())
    if result.returncode != 0:
        if result.stderr.strip():
            print(result.stderr.strip())
        if check:
            raise RuntimeError(f"Command failed ({result.returncode}): {' '.join(map(str, cmd))}")
    return result

if IN_COLAB:
    if not REPO_DIR.exists():
        REPO_DIR.parent.mkdir(parents=True, exist_ok=True)
        run_cmd(["git", "clone", "--branch", REPO_BRANCH, "--single-branch", REPO_URL, str(REPO_DIR)])
    else:
        if not (REPO_DIR / ".git").exists():
            raise RuntimeError(
                f"{REPO_DIR} exists but is not a git checkout. Set NEUROVLM_REPO_DIR to a clean path "
                "or remove that folder, then rerun this cell."
            )
        run_cmd(["git", "-C", str(REPO_DIR), "fetch", "origin", REPO_BRANCH])
        checkout = run_cmd(["git", "-C", str(REPO_DIR), "checkout", REPO_BRANCH], check=False)
        if checkout.returncode != 0:
            run_cmd(["git", "-C", str(REPO_DIR), "checkout", "-B", REPO_BRANCH, f"origin/{REPO_BRANCH}"])
        run_cmd(["git", "-C", str(REPO_DIR), "pull", "--ff-only", "origin", REPO_BRANCH])

os.chdir(REPO_DIR)

if INSTALL_DEPENDENCIES and IN_COLAB:
    run_cmd([sys.executable, "-m", "pip", "install", "-q", "nilearn", "nibabel", "huggingface-hub", "safetensors", "adapters", "transformers", "pyarrow", "matplotlib", "pandas", "scikit-learn", "tqdm", "umap-learn"])
    run_cmd([sys.executable, "-m", "pip", "install", "-q", "-e", ".[viz,notebook,metrics]"])

sys.path.insert(0, str(REPO_DIR / "experiments" / "3dcnn"))
sys.path.insert(0, str(REPO_DIR / "src"))
sys.path.insert(0, str(REPO_DIR))

print("Working directory:", os.getcwd())
if (REPO_DIR / ".git").exists():
    print("Repo branch:", run_cmd(["git", "branch", "--show-current"], cwd=REPO_DIR, check=False).stdout.strip())
print("Drive root:", DRIVE_ROOT)
print("LOCK_DOWNSTREAM_STAGE1_SELECTION:", LOCK_DOWNSTREAM_STAGE1_SELECTION)
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)

# The evaluator lives next to this notebook in experiments/3dcnn.
evaluator_path = REPO_DIR / "experiments/3dcnn/atlas_free_cnn/evaluation/stage1_checkpoint_evaluation.py"
if not evaluator_path.exists():
    raise FileNotFoundError(
        f"Missing {evaluator_path}. Pull the version of the repo that includes "
        "experiments/3dcnn/atlas_free_cnn/evaluation/stage1_checkpoint_evaluation.py, or upload that file next to this notebook."
    )

from atlas_free_cnn.evaluation.stage1_checkpoint_evaluation import EvaluationConfig, run_evaluation
from atlas_free_cnn.notebook_utils import (
    LOCKED_STAGE1_CHECKPOINT_NAMES,
    locked_stage1_checkpoint_selection,
)

# Set this to the completed notebook-6 ablation run directory.
# Example: /content/drive/MyDrive/neurovlm/runs_atlas_free_cnn_ae_ablation/ae_ablation_20260623_123456
RUN_ROOT_VALUE = os.environ.get("NEUROVLM_AE_ABLATION_RUN_DIR", "").strip()
RUN_ROOT = Path(RUN_ROOT_VALUE).expanduser() if RUN_ROOT_VALUE else DRIVE_ROOT / "runs_atlas_free_cnn_ae_ablation/EDIT_ME_AE_ABLATION_RUN_DIR"
print("AE ablation run root:", RUN_ROOT)
if not RUN_ROOT_VALUE:
    print("Set NEUROVLM_AE_ABLATION_RUN_DIR or edit RUN_ROOT/run_dir entries before running evaluation.")
    candidate_root = DRIVE_ROOT / "runs_atlas_free_cnn_ae_ablation"
    candidates = sorted(candidate_root.glob("ae_ablation_*")) if candidate_root.exists() else []
    print("Candidate ablation run directories:")
    for candidate in candidates:
        print("-", candidate)

# If your actual paths differ, edit the run_dir strings directly here. Do not rely on mtime/order discovery.
# Notebook 5 writes Stage 1B directly under 02_stage1b_ae_finetuning/<domain>/checkpoints.
# Older/alternate drafts may have used 02_stage1b_domain_finetune/<domain>/<variant>/checkpoints.
def stage1b_run_dir(domain: str) -> Path:
    candidates = [
        RUN_ROOT / "02_stage1b_ae_finetuning" / domain,
        RUN_ROOT / "02_stage1b_domain_finetune" / domain,
        RUN_ROOT / "02_stage1b_domain_finetune" / domain / f"mixed_baseline_to_{domain}",
        RUN_ROOT / "02_stage1b_domain_finetune" / domain / f"mixed_to_{domain}",
    ]
    for candidate in candidates:
        if (candidate / "checkpoints").exists() or any(candidate.glob("*.pt")):
            return candidate
    return candidates[0]

AE_RUN_REGISTRY = {
    "mixed_baseline_raw_mse": {
        "run_dir": str(RUN_ROOT / "01_stage1_ae_pretraining/mixed_baseline_raw_mse"),
        "stage": "stage1a",
        "training_domain": "mixed",
        "test_domains": ["mixed", "pubmed", "nilearn", "neurovault"],
    },
    "mixed_balanced_raw_mse": {
        "run_dir": str(RUN_ROOT / "01_stage1_ae_pretraining/mixed_balanced_raw_mse"),
        "stage": "stage1a",
        "training_domain": "mixed",
        "test_domains": ["mixed", "pubmed", "nilearn", "neurovault"],
    },
    "mixed_balanced_hybrid_loss": {
        "run_dir": str(RUN_ROOT / "01_stage1_ae_pretraining/mixed_balanced_hybrid_loss"),
        "stage": "stage1a",
        "training_domain": "mixed",
        "test_domains": ["mixed", "pubmed", "nilearn", "neurovault"],
    },
    "mixed_to_pubmed": {
        "run_dir": str(stage1b_run_dir("pubmed")),
        "stage": "stage1b",
        "training_domain": "pubmed",
        "test_domains": ["pubmed"],
        "cross_domain_test_domains": ["mixed", "nilearn", "neurovault"],
    },
    "mixed_to_nilearn": {
        "run_dir": str(stage1b_run_dir("nilearn")),
        "stage": "stage1b",
        "training_domain": "nilearn",
        "test_domains": ["nilearn"],
        "cross_domain_test_domains": ["mixed", "pubmed", "neurovault"],
    },
    "mixed_to_neurovault": {
        "run_dir": str(stage1b_run_dir("neurovault")),
        "stage": "stage1b",
        "training_domain": "neurovault",
        "test_domains": ["neurovault"],
        "cross_domain_test_domains": ["mixed", "pubmed", "nilearn"],
    },
}

print("Stage 1B domain directories found:")
for domain in ["pubmed", "nilearn", "neurovault"]:
    for parent in ["02_stage1b_ae_finetuning", "02_stage1b_domain_finetune"]:
        domain_root = RUN_ROOT / parent / domain
        print(f"{parent}/{domain}:", sorted(str(p) for p in domain_root.iterdir()) if domain_root.exists() else "MISSING")

TEST_JSONL = os.environ.get("NEUROVLM_TEST_JSONL", "")  # leave blank to use the repo fixed test split
OUTPUT_ROOT = Path(os.environ.get("NEUROVLM_AE_EVAL_OUTPUT_ROOT", DRIVE_ROOT / "runs_atlas_free_cnn_checkpoint_evaluation")).expanduser()

EVAL_BATCH_SIZE = int(os.environ.get("NEUROVLM_AE_EVAL_BATCH_SIZE", "32"))
EVAL_NUM_WORKERS = int(os.environ.get("NEUROVLM_EVAL_NUM_WORKERS", "2" if IN_COLAB else "0"))
OVERWRITE = False
MAKE_QUALITATIVE_PLOTS = True
EVALUATE_STAGE1B_CROSS_DOMAIN = True

# Stage 1A recipe comparison must evaluate every configured Stage 1A recipe.
# Missing Stage 1A recipes fail loudly by default; set to 0 to keep them in the output tables as missing rows.
REQUIRE_ALL_STAGE1A_RECIPES = os.environ.get("NEUROVLM_REQUIRE_ALL_STAGE1A_RECIPES", "1") == "1"

# Keep False when Stage 1A and Stage 1B were completed in different notebook runs.
# Missing Stage 1B configured variants will be reported and skipped; available checkpoint dirs still run.
REQUIRE_ALL_REGISTRY_CHECKPOINTS = False

AE_RUN_REGISTRY


In [ ]:
def checkpoint_dir_for_registry_path(run_dir: Path) -> Path:
    return run_dir / "checkpoints" if (run_dir / "checkpoints").exists() else run_dir

STAGE1A_RECIPE_VARIANTS = {variant for variant, spec in AE_RUN_REGISTRY.items() if spec.get("stage") == "stage1a"}
missing_or_empty = []
missing_stage1a = []
ACTIVE_AE_RUN_REGISTRY = {}
for variant, spec in AE_RUN_REGISTRY.items():
    run_dir = Path(spec["run_dir"])
    ckpt_dir = checkpoint_dir_for_registry_path(run_dir)
    pt_files = sorted(p.name for p in ckpt_dir.glob("*.pt")) if ckpt_dir.exists() else []
    is_missing = "EDIT_ME_AE_ABLATION_RUN_DIR" in str(run_dir) or not run_dir.exists() or not pt_files
    if is_missing:
        missing_or_empty.append((variant, run_dir, ckpt_dir, pt_files))
        if variant in STAGE1A_RECIPE_VARIANTS:
            missing_stage1a.append((variant, run_dir, ckpt_dir, pt_files))
            ACTIVE_AE_RUN_REGISTRY[variant] = spec
    else:
        ACTIVE_AE_RUN_REGISTRY[variant] = spec

if missing_or_empty:
    print("Some configured checkpoint directories are missing or empty.")
    print("Current RUN_ROOT:", RUN_ROOT)
    for variant, run_dir, ckpt_dir, pt_files in missing_or_empty:
        print(f"- SKIP {variant}: run_dir={run_dir} exists={run_dir.exists()} checkpoint_dir={ckpt_dir} pt_files={pt_files[:8]}")

print("\nActive checkpoint directories:")
for variant, spec in ACTIVE_AE_RUN_REGISTRY.items():
    run_dir = Path(spec["run_dir"])
    ckpt_dir = checkpoint_dir_for_registry_path(run_dir)
    pt_files = sorted(p.name for p in ckpt_dir.glob("*.pt"))
    print(f"- RUN {variant}: {run_dir} ({len(pt_files)} .pt files)")

if REQUIRE_ALL_STAGE1A_RECIPES and missing_stage1a:
    raise FileNotFoundError("Strict Stage 1A recipe check is enabled and one or more Stage 1A recipe checkpoint directories are missing or empty.")
if REQUIRE_ALL_REGISTRY_CHECKPOINTS and missing_or_empty:
    raise FileNotFoundError("Strict registry check is enabled and one or more configured checkpoint directories are missing or empty.")
if not ACTIVE_AE_RUN_REGISTRY:
    raise FileNotFoundError("No evaluation run started because none of the configured checkpoint directories contain .pt files.")

cfg = EvaluationConfig(
    registry=ACTIVE_AE_RUN_REGISTRY,
    output_root=OUTPUT_ROOT,
    test_jsonl=Path(TEST_JSONL).expanduser() if TEST_JSONL else None,
    device="auto",
    eval_batch_size=EVAL_BATCH_SIZE,
    num_workers=EVAL_NUM_WORKERS,
    overwrite=OVERWRITE,
    make_qualitative_plots=MAKE_QUALITATIVE_PLOTS,
    evaluate_stage1b_cross_domain=EVALUATE_STAGE1B_CROSS_DOMAIN,
)

EVAL_OUTPUT_DIR = run_evaluation(cfg)
EVAL_OUTPUT_DIR


In [ ]:
import json
import pandas as pd

selection_dir = EVAL_OUTPUT_DIR / "04_final_selection"
selected_path = selection_dir / "selected_stage2_checkpoints.json"
empirical_path = selection_dir / "empirical_checkpoint_selection.json"
locked_path = selection_dir / "locked_downstream_checkpoint_selection.json"
manifest_path = EVAL_OUTPUT_DIR / "00_metadata/checkpoint_manifest.csv"
stage1a_all_path = EVAL_OUTPUT_DIR / "01_stage1a/stage1a_all_checkpoint_eval.csv"
stage1a_recipe_best_path = EVAL_OUTPUT_DIR / "01_stage1a/stage1a_recipe_best_checkpoint_comparison.csv"
status_path = EVAL_OUTPUT_DIR / "00_metadata/run_status.json"
config_path = EVAL_OUTPUT_DIR / "00_metadata/evaluation_config.json"
warning_path = selection_dir / "locked_vs_empirical_checkpoint_selection_warning.json"

with selected_path.open() as f:
    empirical_selected = json.load(f)
with status_path.open() as f:
    status = json.load(f)
with config_path.open() as f:
    eval_config = json.load(f)

locked_selected = locked_stage1_checkpoint_selection(AE_RUN_REGISTRY)
selection_dir.mkdir(parents=True, exist_ok=True)
with empirical_path.open("w") as f:
    json.dump(empirical_selected, f, indent=2, sort_keys=True)
with locked_path.open("w") as f:
    json.dump(locked_selected, f, indent=2, sort_keys=True)

selection_differences = []
empirical_to_locked_key = {
    "mixed_stage1a": "mixed_stage1a",
    "pubmed_stage1b": "mixed_to_pubmed_stage1b",
    "nilearn_stage1b": "mixed_to_nilearn_stage1b",
    "neurovault_stage1b": "mixed_to_neurovault_stage1b",
}
for empirical_key, locked_key in empirical_to_locked_key.items():
    empirical_name = (empirical_selected.get(empirical_key) or {}).get("checkpoint_name", "")
    locked_name = (locked_selected.get(locked_key) or {}).get("checkpoint_name", "")
    if empirical_name and locked_name and empirical_name != locked_name:
        selection_differences.append({
            "empirical_key": empirical_key,
            "locked_key": locked_key,
            "empirical_checkpoint_name": empirical_name,
            "locked_checkpoint_name": locked_name,
        })

if LOCK_DOWNSTREAM_STAGE1_SELECTION:
    with selected_path.open("w") as f:
        json.dump(locked_selected, f, indent=2, sort_keys=True)
    selected = locked_selected
else:
    selected = empirical_selected

with warning_path.open("w") as f:
    json.dump({
        "LOCK_DOWNSTREAM_STAGE1_SELECTION": LOCK_DOWNSTREAM_STAGE1_SELECTION,
        "locked_manifest": str(locked_path),
        "empirical_manifest": str(empirical_path),
        "backward_compatible_selected_manifest": str(selected_path),
        "differences": selection_differences,
        "action": "locked_manifest_kept" if LOCK_DOWNSTREAM_STAGE1_SELECTION else "empirical_manifest_selected_by_flag",
    }, f, indent=2, sort_keys=True)

print("Locked downstream checkpoint manifest:", locked_path)
print("Empirical checkpoint selection:", empirical_path)
print("Backward-compatible selected manifest:", selected_path)
if selection_differences:
    print("WARNING: empirical selection differs from locked downstream selection; not switching while lock is enabled.")
    for row in selection_differences:
        print("-", row)

print("Configured run dirs:")
for name, spec in eval_config["registry"].items():
    run_dir = Path(spec["run_dir"])
    ckpt_dir = run_dir / "checkpoints" if (run_dir / "checkpoints").exists() else run_dir
    found = sorted(p.name for p in ckpt_dir.glob("*.pt")) if ckpt_dir.exists() else []
    print(f"- {name}: {run_dir}")
    print(f"  exists={run_dir.exists()} checkpoint_dir={ckpt_dir} pt_files={found[:12]}")

print("\nAccepted downstream checkpoint names:")
for key, expected_name in LOCKED_STAGE1_CHECKPOINT_NAMES.items():
    row = locked_selected[key]
    print(f"- {key}: {row['checkpoint_name']} exists={row['exists']} sha256={row['sha256'][:12] if row['sha256'] else ''}")
    if row["checkpoint_name"] != expected_name:
        raise RuntimeError(f"Locked checkpoint mismatch for {key}: {row['checkpoint_name']} != {expected_name}")

required_stage1a_recipe_best_columns = [
    "recipe", "best_checkpoint_name", "best_checkpoint_path", "selection_metric", "selection_metric_value",
    "mse", "foreground_mse", "spatial_corr", "top1_dice", "top5_dice", "top10_dice",
    "epoch", "heldout_split_fingerprint",
]
for csv_path in [stage1a_all_path, stage1a_recipe_best_path]:
    if not csv_path.exists():
        raise FileNotFoundError(f"Missing expected Stage 1A recipe output CSV: {csv_path}")

if stage1a_recipe_best_path.stat().st_size:
    recipe_best_df = pd.read_csv(stage1a_recipe_best_path)
    missing_cols = [c for c in required_stage1a_recipe_best_columns if c not in recipe_best_df.columns]
    if missing_cols:
        raise RuntimeError(f"Stage 1A recipe-best table is missing required columns: {missing_cols}")
    print("\nStage 1A recipe-best checkpoint comparison:", stage1a_recipe_best_path)
    display(recipe_best_df[required_stage1a_recipe_best_columns])
else:
    print("\nStage 1A recipe-best checkpoint comparison is empty:", stage1a_recipe_best_path)

if stage1a_all_path.stat().st_size:
    stage1a_all_df = pd.read_csv(stage1a_all_path)
    print("\nStage 1A all-checkpoint evaluation:", stage1a_all_path)
    display(stage1a_all_df[[c for c in ["recipe", "checkpoint_name", "status", "rank_within_recipe", "is_recipe_best", "selection_metric", "selection_metric_value", "mse", "foreground_mse", "spatial_corr", "top1_dice", "top5_dice", "top10_dice", "epoch", "heldout_split_fingerprint", "checkpoint_path", "error_message"] if c in stage1a_all_df.columns]])
else:
    print("\nStage 1A all-checkpoint evaluation is empty:", stage1a_all_path)

print("\nRun status summary:")
for key in ["variants_configured", "checkpoint_rows", "unique_model_states", "aliases_detected"]:
    print(f"- {key}: {status.get(key)}")
print(f"- load_failures: {len(status.get('load_failures', []))}")

if manifest_path.exists() and manifest_path.stat().st_size:
    manifest_df = pd.read_csv(manifest_path)
    display(manifest_df[[c for c in ["variant", "checkpoint_name", "checkpoint_path", "checkpoint_epoch", "load_status", "error_message"] if c in manifest_df.columns]])
else:
    print("\ncheckpoint_manifest.csv is empty. The configured run dirs did not contain the requested .pt checkpoint names.")

selected
